# 🧪 Lab 5: The Starship Blueprint Audit (Environment Configuration Reality Check)

In this laboratory session, we audit Apache Spark's configuration engine to observe how explicit properties, dynamic session settings, and systemic environment variables register inside the running UI.

**Objective:** Identify the difference between explicitly specified configurations and implicit defaults, verify startup-only vs runtime-changeable settings, and inspect the actual Spark application blueprint through the Spark Web UI.

### Step 1: Initialize the Telemetry Scanner
We spin up our local Spark session and explicitly override several SQL and UI diagnostic parameters to watch them clear our system gates.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os
import time

existing = SparkSession.getActiveSession()
if existing is not None:
    existing.stop()

spark = (
    SparkSession.builder
                .appName("spark-ui-environment-starship-blueprint-lab-05")
                .master("local[*]")
                .config("spark.ui.enabled", "true")
                .config("spark.ui.port", "4040")
                .config("spark.port.maxRetries", "16")
                .config("spark.sql.adaptive.enabled", "false")
                .config("spark.sql.shuffle.partitions", "314")
                .config("spark.executor.memory", "1g")
                .getOrCreate()
)

print(f"Active Spark version: {spark.version}")
print(f"🛰️ Actual Spark UI: {spark.sparkContext.uiWebUrl}")

Active Spark version: 4.1.2
🛰️ Actual Spark UI: http://T14-PF4WM3XL.sdggroup.local:4040


### Step 2: The Delayed Configuration Paradox (The Ignored Command)
We attempt to change a startup/resource-level property, `spark.executor.memory`, after the SparkContext already exists. Spark should reject this change. The point is not that Spark silently ignores the setting; the point is that some deployment/resource properties must be set before the application starts.

**Mission Directive:** Run the cell below, then check your Environment tab inside the Spark Web UI. You should see Spark reject the mid-flight change instead of resizing executor memory after the SparkContext already exists.

In [2]:
# Step 2: Catching the Paradox to Protect the Bridge
try:
    print('Attempting to alter structural resource allocations after system launch...')
    spark.conf.set('spark.executor.memory', '32g')
except Exception as e:
    print('\n🚨 Red Alert! Spark Security Grid Activated!')
    print(f'Captured Expected Exception: {e}')
    print('\n⚡ Didactic Punchline: The engine room successfully blocked the command.')
    print('You cannot rebuild the warp core hull size while flying at Warp 9.')

# Fallback check to read the real, un-mutated baseline value
active_heap_request = spark.sparkContext.getConf().get('spark.executor.memory', 'Default / Not Specified')
print(f'\n🛡️ Actual active cluster footprint: {active_heap_request}')

Attempting to alter structural resource allocations after system launch...

🚨 Red Alert! Spark Security Grid Activated!
Captured Expected Exception: [CANNOT_MODIFY_CONFIG] Cannot modify the value of the Spark config: "spark.executor.memory".
See also 'https://spark.apache.org/docs/latest/sql-migration-guide.html#ddl-statements'. SQLSTATE: 46110

⚡ Didactic Punchline: The engine room successfully blocked the command.
You cannot rebuild the warp core hull size while flying at Warp 9.

🛡️ Actual active cluster footprint: 1g


### Step 3: Execute Runtime Dynamic SQL Overrides
We alter a loose SQL runtime flag (`spark.sql.shuffle.partitions`) and trigger an active wide aggregation loop to verify how dynamic context configurations alter downstream task counts in real time.

In [3]:
spark.sparkContext.setJobDescription('🧬 Blueprint Audit: Dynamic Partition Realignment')

# Modifying a valid runtime configuration parameter
spark.conf.set('spark.sql.shuffle.partitions', '42')

mock_stream = spark.range(0, 1000000).withColumn('key_group', F.col('id') % 10)
aggregation_track = mock_stream.groupBy('key_group').count()

start_clock = time.time()
result_rows = aggregation_track.collect()
final_count = len(result_rows)
end_clock = time.time()

print(f"🎯 Aggregation calculated: {final_count} group summaries collected.")
print(f'⏱️ Core processing runtime: {end_clock - start_clock:.2f} seconds.')
print("⏳ Navigate to the Stages tab now.")
print("With AQE disabled, the shuffle stage should normally show 42 shuffle partitions/tasks.")

🎯 Aggregation calculated: 10 group summaries collected.
⏱️ Core processing runtime: 3.24 seconds.
⏳ Navigate to the Stages tab now.
With AQE disabled, the shuffle stage should normally show 42 shuffle partitions/tasks.


### Step 4: Interrogate Absent Defaults Telemetry
We extract an implicit system property that was never declared inside our initialization parameters to show how missing fields act as structural clues.

In [4]:
# Querying a property that was not explicitly set in Step 1
implicit_speculation = spark.sparkContext.getConf().get('spark.speculation', 'NOT_EXPLICITLY_SET')
print(f'📡 Speculation Blueprint Signal: {implicit_speculation}')

print(f"\n⏳ Explore the Environment tab now: {spark.sparkContext.uiWebUrl}")
input("Press Enter when you are done inspecting the Environment tab and want to stop Spark...")

spark.stop()
print("💀 SparkContext stopped. The Live UI is no longer available.")

📡 Speculation Blueprint Signal: NOT_EXPLICITLY_SET

⏳ Explore the Environment tab now: http://T14-PF4WM3XL.sdggroup.local:4040
💀 SparkContext stopped. The Live UI is no longer available.


## 📊 Post-Lab Analysis: Evidence from the Engine Room

### 1. Startup Properties Are Not Mid-Flight Controls

Step 2 shows that resource/deployment settings such as `spark.executor.memory` must be configured before the Spark application starts. Spark should reject the late change instead of resizing executors mid-flight.

### 2. Runtime SQL Settings Can Affect Later Queries

Step 3 changes `spark.sql.shuffle.partitions` before running a new aggregation. With AQE disabled in this lab, the shuffle stage should normally reflect the configured value of 42 partitions/tasks.

With AQE enabled, do not promise an exact task count: Spark may coalesce shuffle partitions at runtime.

### 3. Missing Spark Properties Usually Mean Defaults

The Environment tab only lists Spark properties that were explicitly set through `spark-defaults.conf`, `SparkConf`, or command-line options. If a Spark config is absent, Spark is usually using its default value.

### ⏱️ Validation Summary

* **Step 1:** Started a local Spark session and printed the actual Spark UI URL.
* **Step 2:** Verified that a startup/resource property cannot be safely changed after SparkContext creation.
* **Step 3:** Changed a mutable SQL runtime setting and observed its effect on a later shuffle.
* **Step 4:** Checked how absent properties reveal implicit defaults.